In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/M5/sales_train_validation.csv")

print(df.shape)

(30490, 1919)


In [2]:
ca_data = df[df["state_id"] == "CA"]

sales_columns = [
    col for col in df.columns
    if col.startswith("d_")
]

ca_daily_demand = ca_data[sales_columns].sum(axis=0)

print("Historical days:", len(ca_daily_demand))
print("Last 7 days:", ca_daily_demand.tail(7).to_numpy())

Historical days: 1913
Last 7 days: [17052 15784 15148 14488 17095 21834 23187]


In [3]:
import sys
import torch
from chronos import BaseChronosPipeline

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Chronos imported successfully!")

c:\Users\upadh\OneDrive\Desktop\TechExpo\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.14.0+cpu
Chronos imported successfully!


In [4]:
import torch
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map="cpu",
    torch_dtype=torch.float32,
)

print("Chronos-Bolt Tiny loaded successfully!")

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 15050.97it/s]

Chronos-Bolt Tiny loaded successfully!


In [5]:
context = torch.tensor(
    ca_daily_demand.to_numpy(),
    dtype=torch.float32
)

print("Input shape:", context.shape)
print("Last 7 historical values:")
print(context[-7:])

Input shape: torch.Size([1913])
Last 7 historical values:
tensor([17052., 15784., 15148., 14488., 17095., 21834., 23187.])


In [6]:
quantiles, mean = pipeline.predict_quantiles(
    inputs=context,
    prediction_length=7,
    quantile_levels=[0.1, 0.5, 0.9],
)

In [7]:
print("Quantiles shape:", quantiles.shape)
print("Mean shape:", mean.shape)

Quantiles shape: torch.Size([1, 7, 3])
Mean shape: torch.Size([1, 7])


In [8]:
median_forecast = quantiles[0, :, 1]

print("\nNext 7 days predicted CA demand:")

for day, demand in enumerate(median_forecast, start=1):
    print(f"Day {day}: {demand.item():.0f} units")


Next 7 days predicted CA demand:
Day 1: 18156 units
Day 2: 15197 units
Day 3: 14969 units
Day 4: 15757 units
Day 5: 17723 units
Day 6: 21468 units
Day 7: 22943 units


In [9]:
predicted_weekly_demand = median_forecast.sum().item()

print(
    "\nTotal predicted demand for next 7 days:",
    round(predicted_weekly_demand),
    "units"
)


Total predicted demand for next 7 days: 126214 units


In [10]:
predicted_daily_demand = predicted_weekly_demand / 7

print(
    "Average predicted daily demand:",
    round(predicted_daily_demand),
    "units/day"
)

Average predicted daily demand: 18031 units/day


In [11]:
nodes = pd.read_csv("../data/synthetic/fulfillment_nodes.csv")

print(nodes.columns)
print(nodes.head())

Index(['node_id', 'name', 'type', 'status', 'city', 'cluster_id', 'latitude',
       'longitude', 'storage_capacity_units',
       'processing_capacity_units_per_hour', 'operating_hours_per_day',
       'operating_cost_inr_per_day', 'standby_cost_inr_per_day',
       'activation_cost_inr', 'activation_time_hours',
       'max_delivery_radius_km', 'is_elastic'],
      dtype='str')
  node_id               name       type   status         city cluster_id  \
0     N01        NCR Main FC  WAREHOUSE   ACTIVE     Gurugram        NCR   
1     N02        NCR Flex FC   MICRO_FC  STANDBY        Delhi        NCR   
2     N03     Mumbai Main FC  WAREHOUSE   ACTIVE        Thane        MUM   
3     N04     Mumbai Flex FC   MICRO_FC  STANDBY  Navi Mumbai        MUM   
4     N05  Bengaluru Main FC  WAREHOUSE   ACTIVE    Bengaluru        BLR   

   latitude  longitude  storage_capacity_units  \
0     28.49      77.04                    4000   
1     28.52      77.13                    2000   
2     19.2

In [12]:
nodes["daily_processing_capacity"] = (
    nodes["processing_capacity_units_per_hour"]
    * nodes["operating_hours_per_day"]
)

print(
    nodes[
        [
            "node_id",
            "name",
            "cluster_id",
            "status",
            "daily_processing_capacity"
        ]
    ]
)

  node_id               name cluster_id   status  daily_processing_capacity
0     N01        NCR Main FC        NCR   ACTIVE                        320
1     N02        NCR Flex FC        NCR  STANDBY                        320
2     N03     Mumbai Main FC        MUM   ACTIVE                        320
3     N04     Mumbai Flex FC        MUM  STANDBY                        320
4     N05  Bengaluru Main FC        BLR   ACTIVE                        320
5     N06  Bengaluru Flex FC        BLR  STANDBY                        320


In [13]:
ncr_nodes = nodes[nodes["cluster_id"] == "NCR"]

print(ncr_nodes[
    [
        "node_id",
        "name",
        "status",
        "daily_processing_capacity",
        "activation_cost_inr"
    ]
])

  node_id         name   status  daily_processing_capacity  \
0     N01  NCR Main FC   ACTIVE                        320   
1     N02  NCR Flex FC  STANDBY                        320   

   activation_cost_inr  
0                    0  
1                 1500  


In [14]:
active_ncr = ncr_nodes[
    ncr_nodes["status"] == "ACTIVE"
]

active_capacity = active_ncr[
    "daily_processing_capacity"
].sum()

print("Current NCR active capacity:", active_capacity, "units/day")

Current NCR active capacity: 320 units/day


In [15]:
DEMAND_SCALE_FACTOR = 0.018

In [16]:
scaled_forecast = median_forecast * DEMAND_SCALE_FACTOR

print("Scaled NCR forecast:\n")

for day, demand in enumerate(scaled_forecast, start=1):
    print(f"Day {day}: {demand.item():.0f} units")

Scaled NCR forecast:

Day 1: 327 units
Day 2: 274 units
Day 3: 269 units
Day 4: 284 units
Day 5: 319 units
Day 6: 386 units
Day 7: 413 units


In [17]:
simulation_nodes = ncr_nodes.copy()

In [18]:
def get_active_capacity(nodes):
    active_nodes = nodes[nodes["status"] == "ACTIVE"]

    return active_nodes["daily_processing_capacity"].sum()

In [19]:
def get_base_capacity(nodes):
    """
    Capacity provided by non-elastic permanent warehouses.
    """

    base_nodes = nodes[
        (nodes["status"] == "ACTIVE")
        & (nodes["is_elastic"] == False)
    ]

    return base_nodes["daily_processing_capacity"].sum()

In [20]:
# ============================================================
# RESET SIMULATION
# ============================================================

# Every time you run this cell, the simulation starts again
# from the original NCR node states:
# N01 = ACTIVE
# N02 = STANDBY

simulation_nodes = ncr_nodes.copy()

print("Starting network state:")
print(
    simulation_nodes[
        ["node_id", "name", "status"]
    ]
)


# ============================================================
# SIMULATION SETTINGS
# ============================================================

target_utilization = 0.85

# Used to prevent rapid SCALE IN / SCALE OUT
low_demand_days = 0


# ============================================================
# 7-DAY SIMULATION
# ============================================================

for day, forecast in enumerate(scaled_forecast, start=1):

    demand = forecast.item()

    # Recalculate active capacity every day because
    # nodes may have been activated/deactivated
    active_capacity = get_active_capacity(simulation_nodes)

    # We don't want warehouses operating at 100% capacity
    required_capacity = demand / target_utilization

    print(f"\n========== DAY {day} ==========")

    print("Forecast demand:", round(demand), "units")
    print(
        "Required capacity:",
        round(required_capacity),
        "units/day"
    )
    print(
        "Current active capacity:",
        active_capacity,
        "units/day"
    )


    # ========================================================
    # CASE 1: NOT ENOUGH CAPACITY → SCALE OUT
    # ========================================================

    if required_capacity > active_capacity:

        # Demand is high again, so reset low-demand counter
        low_demand_days = 0

        print("\n⚠ Capacity shortage detected")
        print("Decision: SCALE OUT")

        # Find elastic warehouses currently on standby
        standby_nodes = simulation_nodes[
            (simulation_nodes["status"] == "STANDBY")
            & (simulation_nodes["is_elastic"] == True)
        ]

        if not standby_nodes.empty:

            # Choose cheapest warehouse to activate
            selected_index = standby_nodes[
                "activation_cost_inr"
            ].idxmin()

            selected_node = simulation_nodes.loc[selected_index]

            print("\nActivating:")
            print(
                selected_node["node_id"],
                "-",
                selected_node["name"]
            )

            print(
                "Activation cost: ₹",
                selected_node["activation_cost_inr"]
            )

            # Actually activate the warehouse
            simulation_nodes.loc[
                selected_index,
                "status"
            ] = "ACTIVE"

            # Recalculate capacity
            new_capacity = get_active_capacity(
                simulation_nodes
            )

            print("Status: STANDBY → ACTIVE")

            print(
                "New network capacity:",
                new_capacity,
                "units/day"
            )

        else:

            print(
                "❌ No standby elastic node available!"
            )


    # ========================================================
    # CASE 2: CAPACITY IS CURRENTLY SUFFICIENT
    # ========================================================

    else:

        # Capacity of permanent/non-elastic warehouses only
        base_capacity = get_base_capacity(
            simulation_nodes
        )

        # Find elastic warehouses that are currently active
        active_elastic_nodes = simulation_nodes[
            (simulation_nodes["status"] == "ACTIVE")
            & (simulation_nodes["is_elastic"] == True)
        ]


        # ----------------------------------------------------
        # Check whether permanent warehouse alone can handle
        # the forecast
        # ----------------------------------------------------

        if required_capacity <= base_capacity:

            low_demand_days += 1

        else:

            low_demand_days = 0


        # ----------------------------------------------------
        # SCALE IN only after 2 consecutive low-demand days
        # ----------------------------------------------------

        if (
            low_demand_days >= 2
            and not active_elastic_nodes.empty
        ):

            print("\nLow demand sustained for 2 days")
            print("Decision: SCALE IN")

            selected_index = active_elastic_nodes.index[0]

            selected_node = simulation_nodes.loc[
                selected_index
            ]

            print(
                "Deactivating:",
                selected_node["node_id"],
                "-",
                selected_node["name"]
            )

            # Actually deactivate warehouse
            simulation_nodes.loc[
                selected_index,
                "status"
            ] = "STANDBY"

            # Reset counter after scaling in
            low_demand_days = 0

            new_capacity = get_active_capacity(
                simulation_nodes
            )

            print("Status: ACTIVE → STANDBY")

            print(
                "New network capacity:",
                new_capacity,
                "units/day"
            )


        # ----------------------------------------------------
        # Otherwise don't change the network
        # ----------------------------------------------------

        else:

            print("\n✓ Capacity sufficient")
            print("Decision: MAINTAIN")


# ============================================================
# FINAL NETWORK STATE
# ============================================================

print("\n========== FINAL NETWORK ==========")

print(
    simulation_nodes[
        [
            "node_id",
            "name",
            "status",
            "daily_processing_capacity"
        ]
    ]
)

Starting network state:
  node_id         name   status
0     N01  NCR Main FC   ACTIVE
1     N02  NCR Flex FC  STANDBY

========== DAY 1 ==========
Forecast demand: 327 units
Required capacity: 384 units/day
Current active capacity: 320 units/day

⚠ Capacity shortage detected
Decision: SCALE OUT

Activating:
N02 - NCR Flex FC
Activation cost: ₹ 1500
Status: STANDBY → ACTIVE
New network capacity: 640 units/day

========== DAY 2 ==========
Forecast demand: 274 units
Required capacity: 322 units/day
Current active capacity: 640 units/day

✓ Capacity sufficient
Decision: MAINTAIN

========== DAY 3 ==========
Forecast demand: 269 units
Required capacity: 317 units/day
Current active capacity: 640 units/day

✓ Capacity sufficient
Decision: MAINTAIN

========== DAY 4 ==========
Forecast demand: 284 units
Required capacity: 334 units/day
Current active capacity: 640 units/day

✓ Capacity sufficient
Decision: MAINTAIN

========== DAY 5 ==========
Forecast demand: 319 units
Required capacity: 3

In [21]:
inventory = pd.read_csv("../data/synthetic/inventory.csv")

print(inventory.columns)
print(inventory.head())

Index(['node_id', 'product_id', 'available_qty', 'reserved_qty',
       'in_transit_qty', 'safety_stock', 'last_updated', 'version'],
      dtype='str')
  node_id product_id  available_qty  reserved_qty  in_transit_qty  \
0     N01        P01            250             0               0   
1     N01        P02            250             0               0   
2     N01        P03            250             0               0   
3     N01        P04            250             0               0   
4     N01        P05            250             0               0   

   safety_stock                                       last_updated  version  
0            20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
1            20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
2            20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
3            20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
4            20  Sat Sep 19 2026 00:00:00 

In [22]:
n02_inventory = inventory[
    inventory["node_id"] == "N02"
]

print(n02_inventory)

print(
    "\nTotal available inventory:",
    n02_inventory["available_qty"].sum()
)

print(
    "Total reserved inventory:",
    n02_inventory["reserved_qty"].sum()
)

print(
    "Total in-transit inventory:",
    n02_inventory["in_transit_qty"].sum()
)

print(
    "Total safety stock:",
    n02_inventory["safety_stock"].sum()
)

   node_id product_id  available_qty  reserved_qty  in_transit_qty  \
6      N02        P01              0             0               0   
7      N02        P02              0             0               0   
8      N02        P03              0             0               0   
9      N02        P04              0             0               0   
10     N02        P05              0             0               0   
11     N02        P06              0             0               0   

    safety_stock                                       last_updated  version  
6             20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
7             20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
8             20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
9             20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
10            20  Sat Sep 19 2026 00:00:00 GMT+0530 (India Stand...        1  
11            20  Sat Sep 19 2026 0

In [23]:
n01_inventory = inventory[
    inventory["node_id"] == "N01"
]

print(
    n01_inventory[
        [
            "product_id",
            "available_qty",
            "reserved_qty",
            "in_transit_qty",
            "safety_stock"
        ]
    ]
)

  product_id  available_qty  reserved_qty  in_transit_qty  safety_stock
0        P01            250             0               0            20
1        P02            250             0               0            20
2        P03            250             0               0            20
3        P04            250             0               0            20
4        P05            250             0               0            20
5        P06            250             0               0            20


In [24]:
n01_inventory = n01_inventory.copy()

n01_inventory["usable_qty"] = (
    n01_inventory["available_qty"]
    - n01_inventory["reserved_qty"]
    - n01_inventory["safety_stock"]
)

print(
    n01_inventory[
        [
            "product_id",
            "available_qty",
            "reserved_qty",
            "safety_stock",
            "usable_qty"
        ]
    ]
)

  product_id  available_qty  reserved_qty  safety_stock  usable_qty
0        P01            250             0            20         230
1        P02            250             0            20         230
2        P03            250             0            20         230
3        P04            250             0            20         230
4        P05            250             0            20         230
5        P06            250             0            20         230


In [25]:
total_usable_inventory = n01_inventory["usable_qty"].sum()

print(
    "Total usable inventory at N01:",
    total_usable_inventory,
    "units"
)

Total usable inventory at N01: 1380 units


In [26]:
day1_demand = round(scaled_forecast[0].item())

n02_share = 0.40

n02_target_demand = round(
    day1_demand * n02_share
)

print("Day 1 regional demand:", day1_demand)
print("N02 target share:", n02_share * 100, "%")
print("Inventory needed at N02:", n02_target_demand)

Day 1 regional demand: 327
N02 target share: 40.0 %
Inventory needed at N02: 131
